# Lego Dataset — Exploratory Data Analysis
Azure SQL → pandas → matplotlib / seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from db import query

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

---
## 1 · Load tables from the database

In [ ]:
sets        = query("SELECT * FROM sets")
themes      = query("SELECT * FROM themes")
colors      = query("SELECT * FROM colors")
parts       = query("SELECT * FROM parts")
part_cats   = query("SELECT * FROM part_categories")
inv_parts   = query("SELECT * FROM inventory_parts")

l = sets['num_parts'] == 0
print(f"There are {len(l)} sets without parts.")

print("sets:",      sets.shape)
print("themes:",    themes.shape)
print("colors:",    colors.shape)
print("parts:",     parts.shape)
print("part_cats:", part_cats.shape)
print("inv_parts:", inv_parts.shape)

In [ ]:
avg_parts_per_theme = sets.groupby('theme_id')['num_parts'].mean().reset_index()
avg_parts_per_theme = avg_parts_per_theme.merge(themes[['id', 'name']], left_on='theme_id', right_on='id', how='left')
avg_parts_per_theme = avg_parts_per_theme[['name', 'num_parts']].rename(columns={'num_parts': 'avg_pieces'})
avg_parts_per_theme = avg_parts_per_theme.sort_values(by='avg_pieces')
display(avg_parts_per_theme)

# Save the dataframe to CSV
avg_parts_per_theme.to_csv('avg_parts_per_theme.csv', index=False)

---
## 2 · Summary statistics & missingness

In [ ]:
print("=== sets ===")
display(sets.describe(include='all'))

print("\n=== themes ===")
display(themes.describe(include='all'))

print("\n=== colors ===")
display(colors.describe(include='all'))

In [ ]:
tables = {'sets': sets, 'themes': themes, 'colors': colors,
          'parts': parts, 'part_cats': part_cats, 'inv_parts': inv_parts}

missingness = pd.DataFrame({
    name: df.isnull().sum() for name, df in tables.items()
}).fillna(0).astype(int)

print("NULL counts per column (0 = no NULLs):")
display(missingness[missingness.sum(axis=1) > 0])  # show only columns with any NULLs

# If nothing prints above, all columns are fully populated
if (missingness.sum(axis=1) > 0).sum() == 0:
    print("  → No missing values detected in any loaded table.")

---
## 3 · Distribution plots

In [ ]:
# 3a — Distribution of num_parts per set (log scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(sets['num_parts'], bins=80, ax=axes[0], color='steelblue')
axes[0].set_title('num_parts distribution (linear scale)')
axes[0].set_xlabel('Number of parts')

sns.histplot(sets[sets['num_parts'] > 0]['num_parts'], bins=80,
             log_scale=(True, False), ax=axes[1], color='coral')
axes[1].set_title('num_parts distribution (log x-axis)')
axes[1].set_xlabel('Number of parts (log scale)')

plt.tight_layout()
plt.savefig('figures/dist_num_parts.png', dpi=150)
plt.show()

In [ ]:
# 3b — Number of sets released per year
sets_per_year = sets.groupby('year').size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(sets_per_year['year'], sets_per_year['count'], color='steelblue', width=0.8)
ax.set_title('Sets released per year')
ax.set_xlabel('Year')
ax.set_ylabel('Number of sets')
ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
plt.tight_layout()
plt.savefig('figures/sets_per_year.png', dpi=150)
plt.show()

In [ ]:
# 3c — Transparent vs opaque color breakdown
color_counts = colors['is_trans'].value_counts().rename({0: 'Opaque', 1: 'Transparent'})

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(color_counts, labels=color_counts.index, autopct='%1.1f%%',
       colors=['#4878cf', '#6acc65'], startangle=90)
ax.set_title('Color transparency breakdown')
plt.tight_layout()
plt.savefig('figures/color_transparency.png', dpi=150)
plt.show()

In [ ]:
# 3d — Top 15 themes by set count
top_themes = (
    sets.groupby('theme_id').size()
        .reset_index(name='set_count')
        .merge(themes[['id', 'name']], left_on='theme_id', right_on='id')
        .nlargest(15, 'set_count')
        .sort_values('set_count')
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_themes['name'], top_themes['set_count'], color='mediumpurple')
ax.set_title('Top 15 themes by set count')
ax.set_xlabel('Number of sets')
plt.tight_layout()
plt.savefig('figures/top_themes.png', dpi=150)
plt.show()

---
## 4 · Relationship analysis

In [ ]:
# 4a — Median set complexity (num_parts) over time
complexity = sets.groupby('year')['num_parts'].median().reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(complexity['year'], complexity['num_parts'], marker='o', markersize=3,
        linewidth=1.5, color='darkorange')
ax.set_title('Median num_parts per set over time')
ax.set_xlabel('Year')
ax.set_ylabel('Median number of parts')
ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
plt.tight_layout()
plt.savefig('figures/complexity_over_time.png', dpi=150)
plt.show()

In [ ]:
# 4b — Pearson correlation: year vs num_parts
corr = sets[['year', 'num_parts']].corr()
print("Correlation matrix (year, num_parts):")
display(corr)

r = corr.loc['year', 'num_parts']
print(f"\nPearson r = {r:.3f}")

In [ ]:
# 4c — num_parts box plot: top 8 themes
top8_ids = (
    sets.groupby('theme_id').size()
        .nlargest(8).index
)
sets_top8 = (
    sets[sets['theme_id'].isin(top8_ids)]
        .merge(
            themes[['id', 'name']].rename(columns={'name': 'theme_name'}),
            left_on='theme_id', right_on='id'
        )
)

order = (
    sets_top8.groupby('theme_name')['num_parts']
             .median().sort_values(ascending=False).index
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=sets_top8, x='theme_name', y='num_parts', order=order,
            showfliers=False, palette='Set2', ax=ax)
ax.set_title('num_parts distribution across top-8 themes (no outliers)')
ax.set_xlabel('Theme')
ax.set_ylabel('Number of parts')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('figures/parts_by_theme.png', dpi=150)
plt.show()

In [ ]:
# 4d — Most-used colors in inventory_parts
color_usage = (
    inv_parts.groupby('color_id')['quantity'].sum()
             .reset_index(name='total_qty')
             .merge(colors[['id', 'name', 'rgb', 'is_trans']], left_on='color_id', right_on='id')
             .nlargest(15, 'total_qty')
             .sort_values('total_qty')
)

# Build hex color list for bars
bar_colors = ['#' + rgb if len(str(rgb)) == 6 else '#888888'
              for rgb in color_usage['rgb']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(color_usage['name'], color_usage['total_qty'], color=bar_colors)
ax.set_title('Top 15 colors by total part quantity in inventories')
ax.set_xlabel('Total quantity across all inventories')
plt.tight_layout()
plt.savefig('figures/top_colors.png', dpi=150)
plt.show()

### 4e · Color breakdown for the top 8 largest sets

In [ ]:
# 4e — Color breakdown for the top 8 sets by num_parts
inventories = query("SELECT * FROM inventories")

top8_sets = sets.nlargest(8, 'num_parts')[['set_num', 'name', 'num_parts']].copy()

# sets → inventories → inv_parts → colors
color_per_set = (
    top8_sets
    .merge(inventories[['id', 'set_num']], on='set_num')
    .rename(columns={'id': 'inventory_id'})
    .merge(inv_parts, on='inventory_id')
    .merge(
        colors[['id', 'name', 'rgb']].rename(columns={'id': 'color_id', 'name': 'color_name'}),
        on='color_id'
    )
    .groupby(['set_num', 'name', 'color_name', 'rgb'], as_index=False)['quantity'].sum()
)

# Keep only the top-N colors per set so bars stay readable; lump the rest as "Other"
TOP_N = 8
def top_n_colors(grp):
    grp = grp.sort_values('quantity', ascending=False)
    top = grp.head(TOP_N)
    other_qty = grp.iloc[TOP_N:]['quantity'].sum()
    if other_qty > 0:
        other_row = top.iloc[[0]].copy()
        other_row['color_name'] = 'Other'
        other_row['rgb']        = 'AAAAAA'
        other_row['quantity']   = other_qty
        top = pd.concat([top, other_row], ignore_index=True)
    return top

color_per_set = (
    color_per_set.groupby(['set_num', 'name'], group_keys=False)
                 .apply(top_n_colors)
)

# Pivot to wide form for stacked bar chart
pivot = (
    color_per_set
    .pivot_table(index=['set_num', 'name'], columns='color_name',
                 values='quantity', aggfunc='sum', fill_value=0)
)

# Short set labels: "Set Name (set_num)"
set_labels = [f"{n}\n({s})" for s, n in pivot.index]

# Build a color map: color_name → hex
color_hex = (
    color_per_set[['color_name', 'rgb']]
    .drop_duplicates()
    .set_index('color_name')['rgb']
    .apply(lambda x: '#' + str(x) if not str(x).startswith('#') else x)
    .to_dict()
)

fig, ax = plt.subplots(figsize=(14, 6))
bottom = [0] * len(pivot)
for col in pivot.columns:
    vals = pivot[col].values
    hex_c = color_hex.get(col, '#AAAAAA')
    # Ensure valid 6-char hex; fall back to grey
    try:
        if len(hex_c.lstrip('#')) != 6:
            raise ValueError
        bars = ax.bar(set_labels, vals, bottom=bottom, label=col, color=hex_c)
    except (ValueError, TypeError):
        bars = ax.bar(set_labels, vals, bottom=bottom, label=col, color='#AAAAAA')
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_title('Color breakdown of the top 8 largest LEGO sets (by num_parts)')
ax.set_xlabel('Set')
ax.set_ylabel('Total part quantity')
ax.legend(title='Color', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('figures/colors_top8_sets.png', dpi=150)
plt.show()

---
## 5 · Key insight

**Set complexity has risen dramatically since the 1990s.**

The median `num_parts` per set remained under ~50 pieces through the 1980s,
then climbed steadily — reaching 150+ pieces in recent decades.
The Pearson correlation between `year` and `num_parts` quantifies this trend.

**Implication for later work:** year should be treated as an important feature
in any model predicting set size or complexity. Theme-level analysis should
also control for the era in which sets were released, since older themes
naturally have lower part counts regardless of their category.

In [ ]:
# Numeric summary of the insight
decade = sets.copy()
decade['decade'] = (decade['year'] // 10) * 10
summary = decade.groupby('decade')['num_parts'].agg(['median', 'mean', 'count'])
summary.columns = ['median_parts', 'mean_parts', 'set_count']
display(summary)